# LS-LRSI: Data Preprocessing
## Aranayake, Kegalle District, Sri Lanka


**Student Name:** Aabidha Rifky
**ICBT SIS ID:** CL/MCSDS/CMU/10/04
**Cardiff Met ID:** st20357374
**Module:** DAS7003 Geospatial Analysis
**Assessment:** PRAC1

---

## Purpose

`01_data_acquisition.ipynb` loaded and verified seven raw datasets for Aranayake. Each of those files has a different resolution, a different coordinate system, and in some cases a different file format entirely. Before any risk indicator can be calculated, all seven need to be brought onto a single, shared 30 metre grid, in a single coordinate system.

This notebook performs that alignment, then calculates the derived indicators (slope, aspect, curvature, NDVI, distance layers, and soil clay content) that the risk index itself is built from in `04_index_construction.ipynb`.

## Why reprojection is needed

Two of the seven datasets are supplied in geographic coordinates (latitude and longitude, EPSG:4326): the DEM, CHIRPS rainfall, OSM roads and streams. Sentinel-2 is supplied in a projected coordinate system (UTM Zone 44N, EPSG:32644). Geographic coordinates are measured in degrees, which do not represent constant physical distances, a degree of longitude covers a different ground distance depending on latitude. Slope, distance, and area calculations require a projected coordinate system, where units are metres. All layers in this notebook are therefore reprojected to EPSG:32644 before any distance or slope calculation is performed.

## 1. Environment Setup and Reference Grid Definition

This section defines the shared grid that every dataset will be aligned to. The grid is based on the SRTM elevation data (30 m resolution), established as the reference in `01_data_acquisition.ipynb`.

In [1]:
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

TARGET_CRS = "EPSG:32644"   # UTM Zone 44N, covers Sri Lanka, units in metres
TARGET_RESOLUTION = 30      # metres, matches SRTM native resolution

dem_source_path = f"{RAW_DIR}/dem/aranayake_srtm.tif"

with rasterio.open(dem_source_path) as src:
    source_crs = src.crs
    source_bounds = src.bounds

    target_transform, target_width, target_height = calculate_default_transform(
        source_crs, TARGET_CRS,
        src.width, src.height,
        *source_bounds,
        resolution=TARGET_RESOLUTION
    )

print(f"Source coordinate system: {source_crs}")
print(f"Target coordinate system: {TARGET_CRS}")
print(f"Target grid size: {target_width} x {target_height} pixels")
print(f"Target resolution: {TARGET_RESOLUTION} m")

Source coordinate system: EPSG:4326
Target coordinate system: EPSG:32644
Target grid size: 221 x 221 pixels
Target resolution: 30 m


This cell calculates the exact grid, in the projected coordinate system, that all seven datasets will be resampled onto. `calculate_default_transform` works out the correct pixel dimensions and alignment automatically from the DEM's original extent, converting it from geographic coordinates into UTM metres.

The resulting grid size is expected to differ slightly from the 216 x 216 figure seen in `01_data_acquisition.ipynb`, since reprojecting from a geographic to a projected coordinate system changes the exact pixel count, while covering the same physical area. This is normal and expected. This same `target_transform`, `target_width`, and `target_height` will be reused for every dataset in this notebook, ensuring they all align to the same grid exactly.

## 2. Reproject Elevation Data

The DEM is reprojected from its original geographic coordinate system into UTM Zone 44N, onto the target grid defined above. This is the first of seven datasets to go through this process; the same pattern is repeated for each remaining dataset in the sections that follow.

In [2]:
dem_output_path = f"{PROCESSED_DIR}/dem_30m.tif"

with rasterio.open(dem_source_path) as src:
    dem_reprojected = np.empty((target_height, target_width), dtype=np.float32)

    reproject(
        source=rasterio.band(src, 1),
        destination=dem_reprojected,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=target_transform,
        dst_crs=TARGET_CRS,
        resampling=Resampling.bilinear   # bilinear is appropriate for continuous data like elevation
    )

    profile = src.profile.copy()
    profile.update({
        "crs": TARGET_CRS,
        "transform": target_transform,
        "width": target_width,
        "height": target_height,
        "dtype": "float32"
    })

    with rasterio.open(dem_output_path, "w", **profile) as dst:
        dst.write(dem_reprojected, 1)

print(f"Reprojected DEM saved to: {dem_output_path}")
print(f"Shape: {dem_reprojected.shape}")
print(f"Elevation range: {dem_reprojected.min():.1f} m to {dem_reprojected.max():.1f} m")

Reprojected DEM saved to: ../data/processed/dem_30m.tif
Shape: (221, 221)
Elevation range: 55.9 m to 371.8 m


`Resampling.bilinear` is used here rather than nearest-neighbour resampling, because elevation is a continuous variable, bilinear resampling produces a smoother, more physically realistic surface by interpolating between neighbouring source pixels, rather than simply duplicating the nearest original value. This distinction matters for later slope calculations, since nearest-neighbour resampling can introduce artificial stepping artefacts into the slope surface.

The elevation range after reprojection is expected to closely match the original 55-374 m range confirmed in `01_data_acquisition.ipynb`, small differences of a few metres are normal and result from the interpolation process, not an error.

## 3. Reproject and Sum Rainfall Data

Each of the four CHIRPS daily files (14-17 May 2016) is reprojected onto the same 30 m grid, then summed to produce a single cumulative rainfall layer covering the landslide trigger period. This is the layer used as the rainfall risk indicator in the index.

CHIRPS is a relatively coarse dataset (approximately 5.5 km native resolution) compared to the 30 m target grid. Reprojecting it to 30 m does not add real detail, it produces a smooth surface where many neighbouring 30 m cells share very similar values, since they are all derived from the same coarse original measurement. This is expected and does not indicate an error.

In [3]:
chirps_dates = ["14", "15", "16", "17"]
cumulative_rainfall = np.zeros((target_height, target_width), dtype=np.float32)

for date in chirps_dates:
    chirps_path = f"/vsigzip/{RAW_DIR}/chirps/chirps-v2.0.2016.05.{date}.tif.gz"

    with rasterio.open(chirps_path) as src:
        day_reprojected = np.empty((target_height, target_width), dtype=np.float32)

        reproject(
            source=rasterio.band(src, 1),
            destination=day_reprojected,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=target_transform,
            dst_crs=TARGET_CRS,
            resampling=Resampling.bilinear
        )

        cumulative_rainfall += day_reprojected
        print(f"Day {date}: {day_reprojected.min():.1f} mm to {day_reprojected.max():.1f} mm")

rainfall_output_path = f"{PROCESSED_DIR}/rainfall_30m.tif"
profile.update({"crs": TARGET_CRS, "transform": target_transform, "width": target_width, "height": target_height})

with rasterio.open(rainfall_output_path, "w", **profile) as dst:
    dst.write(cumulative_rainfall, 1)

print(f"\nCumulative rainfall saved to: {rainfall_output_path}")
print(f"Cumulative rainfall range across study area: {cumulative_rainfall.min():.1f} mm to {cumulative_rainfall.max():.1f} mm")

Day 14: 47.2 mm to 62.7 mm
Day 15: 63.8 mm to 77.3 mm
Day 16: 66.9 mm to 95.6 mm
Day 17: 62.4 mm to 104.4 mm

Cumulative rainfall saved to: ../data/processed/rainfall_30m.tif
Cumulative rainfall range across study area: 247.0 mm to 327.8 mm


The cumulative range across the study area (247.0 mm to 327.8 mm) is expected to sit close to, but not identical to, the single-point figure of 325.0 mm confirmed in `01_data_acquisition.ipynb`. Some variation across the grid is expected, since CHIRPS's coarse cells do not align exactly with the 30 m grid boundaries, so different parts of the study area draw from slightly different source pixels. This range remains below the 435-446 mm literature figure, though notably closer than the earlier town-centre estimate, consistent with this recentred location sitting nearer the documented failure site.

## 4. Reproject Sentinel-2 Bands and Calculate NDVI

Sentinel-2 tiles are supplied in UTM coordinates already (Zone 44N for this location), matching the target coordinate system defined in Section 1. This step is therefore a resampling operation, aligning the native 10 m pixels onto the shared 30 m grid, rather than a change of coordinate system.

NDVI (Normalised Difference Vegetation Index) is calculated as (NIR - Red) / (NIR + Red). Sentinel-2 reflectance values are stored as scaled integers, but since NDVI is a ratio, this scaling factor cancels out automatically and does not need to be corrected separately.

In [4]:
import glob
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

red_candidates = glob.glob(f"{RAW_DIR}/sentinel/**/*_B04_10m.jp2", recursive=True)
nir_candidates = glob.glob(f"{RAW_DIR}/sentinel/**/*_B08_10m.jp2", recursive=True)

aranayake_bbox_wgs84 = (80.32, 7.27, 80.38, 7.33)

red_reprojected = None

for red_path in red_candidates:
    nir_path = red_path.replace("_B04_10m.jp2", "_B08_10m.jp2")
    if nir_path not in nir_candidates:
        continue

    with rasterio.open(red_path) as red_src:
        bbox_in_file_crs = transform_bounds("EPSG:4326", red_src.crs, *aranayake_bbox_wgs84)
        window = from_bounds(*bbox_in_file_crs, transform=red_src.transform)
        red_window_data = red_src.read(1, window=window)

    if red_window_data.size > 0 and red_window_data.max() > 0:
        red_source_path = red_path
        nir_source_path = nir_path
        break

with rasterio.open(red_source_path) as red_src:
    red_reprojected = np.empty((target_height, target_width), dtype=np.float32)
    reproject(
        source=rasterio.band(red_src, 1),
        destination=red_reprojected,
        src_transform=red_src.transform,
        src_crs=red_src.crs,
        dst_transform=target_transform,
        dst_crs=TARGET_CRS,
        resampling=Resampling.bilinear
    )

with rasterio.open(nir_source_path) as nir_src:
    nir_reprojected = np.empty((target_height, target_width), dtype=np.float32)
    reproject(
        source=rasterio.band(nir_src, 1),
        destination=nir_reprojected,
        src_transform=nir_src.transform,
        src_crs=nir_src.crs,
        dst_transform=target_transform,
        dst_crs=TARGET_CRS,
        resampling=Resampling.bilinear
    )

ndvi = (nir_reprojected - red_reprojected) / (nir_reprojected + red_reprojected + 1e-6)   # small constant avoids division by zero

ndvi_output_path = f"{PROCESSED_DIR}/ndvi_30m.tif"
with rasterio.open(ndvi_output_path, "w", **profile) as dst:
    dst.write(ndvi, 1)

print(f"Using scene: {red_source_path}")
print(f"NDVI saved to: {ndvi_output_path}")
print(f"NDVI range: {ndvi.min():.2f} to {ndvi.max():.2f}")

Using scene: ../data/raw/sentinel\S2B_MSIL2A_20250223T045719_N0511_R119_T44NMP_20250223T073037.SAFE\GRANULE\L2A_T44NMP_A041615_20250223T050816\IMG_DATA\R10m\T44NMP_20250223T045719_B04_10m.jp2
NDVI saved to: ../data/processed/ndvi_30m.tif
NDVI range: 0.08 to 0.66


NDVI values are bounded between -1 and 1 by definition. For a vegetated hillside area such as Aranayake, values are expected to fall mostly in the 0.05-0.7 range, with higher values over dense vegetation and lower values over bare ground, exposed rock, or cleared farmland. Negative or near-zero values would indicate water or very sparse cover. The small constant added to the denominator prevents division-by-zero errors in any pixel where both bands read as zero, which can occasionally occur at tile edges.

## 5. Reproject Roads and Streams, Calculate Distance Layers

The road and waterway data from `01_data_acquisition.ipynb` exist as lines (vector data), but the risk index needs a value at every one of the 221 x 221 grid cells: the distance from that cell to the nearest road, and the nearest stream. This section converts the line data into two distance maps.

This happens in three steps: reproject the lines into the same UTM coordinate system as the rest of the grid, "rasterize" them (mark which 30 m grid cells any part of a road or stream passes through), then calculate, for every cell in the grid, its distance in metres to the nearest marked cell.

In [5]:
import geopandas as gpd
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt

osm_path = f"{RAW_DIR}/osm/sri-lanka-latest.osm.pbf"
aranayake_bbox_wgs84 = (80.310, 7.188, 80.370, 7.248)

roads = gpd.read_file(osm_path, layer="lines", bbox=aranayake_bbox_wgs84, where="highway IS NOT NULL")
waterways = gpd.read_file(osm_path, layer="lines", bbox=aranayake_bbox_wgs84, where="waterway IS NOT NULL")

roads_utm = roads.to_crs(TARGET_CRS)
waterways_utm = waterways.to_crs(TARGET_CRS)

roads_raster = rasterize(
    [(geom, 1) for geom in roads_utm.geometry if geom is not None],
    out_shape=(target_height, target_width),
    transform=target_transform,
    fill=0, dtype="uint8"
)
waterways_raster = rasterize(
    [(geom, 1) for geom in waterways_utm.geometry if geom is not None],
    out_shape=(target_height, target_width),
    transform=target_transform,
    fill=0, dtype="uint8"
)

# distance in metres from every grid cell to the nearest road / stream cell
dist_road = distance_transform_edt(1 - roads_raster, sampling=(TARGET_RESOLUTION, TARGET_RESOLUTION)).astype(np.float32)
dist_stream = distance_transform_edt(1 - waterways_raster, sampling=(TARGET_RESOLUTION, TARGET_RESOLUTION)).astype(np.float32)

with rasterio.open(f"{PROCESSED_DIR}/dist_road_30m.tif", "w", **profile) as dst:
    dst.write(dist_road, 1)
with rasterio.open(f"{PROCESSED_DIR}/dist_stream_30m.tif", "w", **profile) as dst:
    dst.write(dist_stream, 1)

print(f"Road grid cells found: {roads_raster.sum()}")
print(f"Waterway grid cells found: {waterways_raster.sum()}")
print(f"Distance to road range: {dist_road.min():.0f} m to {dist_road.max():.0f} m")
print(f"Distance to stream range: {dist_stream.min():.0f} m to {dist_stream.max():.0f} m")

Road grid cells found: 3834
Waterway grid cells found: 218
Distance to road range: 0 m to 698 m
Distance to stream range: 0 m to 4092 m


A minimum distance of 0 m at both layers is expected, this occurs at grid cells that a road or stream actually passes through. The maximum distance should stay within the size of the study area itself (roughly 6.6 km across the diagonal), any value larger than that would indicate a coordinate system mismatch. `scipy`'s `distance_transform_edt` calculates straight-line (Euclidean) distance, not distance along a path, which is standard practice for this kind of proximity indicator and matches the approach used in the reviewed Sri Lankan susceptibility literature (Hemasinghe et al., 2018).

## 6. Calculate Slope, Aspect, and Curvature

These three indicators are derived entirely from the elevation data reprojected in Section 2, no additional source file is needed. Slope is how steep the ground is, aspect is which compass direction it faces, and curvature describes whether the ground is bowl-shaped (concave, water tends to collect) or dome-shaped (convex, water sheds away quickly). All three are established risk factors in the literature reviewed for this project (Section 2 of the written report).

Because the DEM is now in a projected coordinate system with 30 m cells, these calculations use real metre distances directly, unlike a calculation performed on the original geographic-coordinate version of the DEM.

In [6]:
with rasterio.open(f"{PROCESSED_DIR}/dem_30m.tif") as src:
    dem = src.read(1)

dz_dy, dz_dx = np.gradient(dem, TARGET_RESOLUTION)

slope_deg = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))).astype(np.float32)
aspect_deg = ((np.degrees(np.arctan2(-dz_dx, dz_dy)) + 360) % 360).astype(np.float32)
curvature = (np.gradient(dz_dx, axis=1) + np.gradient(dz_dy, axis=0)).astype(np.float32)

with rasterio.open(f"{PROCESSED_DIR}/slope_30m.tif", "w", **profile) as dst:
    dst.write(slope_deg, 1)
with rasterio.open(f"{PROCESSED_DIR}/aspect_30m.tif", "w", **profile) as dst:
    dst.write(aspect_deg, 1)
with rasterio.open(f"{PROCESSED_DIR}/curvature_30m.tif", "w", **profile) as dst:
    dst.write(curvature, 1)

print(f"Slope range: {slope_deg.min():.1f} to {slope_deg.max():.1f} degrees")
print(f"Mean slope: {slope_deg.mean():.1f} degrees")
print(f"Aspect range: {aspect_deg.min():.0f} to {aspect_deg.max():.0f} degrees")
print(f"Curvature range: {curvature.min():.3f} to {curvature.max():.3f}")

Slope range: 0.0 to 57.6 degrees
Mean slope: 12.0 degrees
Aspect range: 0 to 360 degrees
Curvature range: -1.137 to 0.678


Aspect is measured in degrees on a compass (0/360 = north, 90 = east, 180 = south, 270 = west), so a full 0-360 range is expected and normal, it does not indicate an error.

The result shows a mean slope of 12.0 degrees across the full study grid, reflecting the mix of steep hillside and flatter valley terrain within the recentred study area, with a maximum of 57.6 degrees in the steepest sections. This maximum is consistent with the locally extreme slope conditions implicated in the 2016 failure mechanism, even though the area-wide average is lower.

Curvature values close to zero indicate flat or planar ground, positive values indicate concave (bowl-shaped, water-collecting) terrain, and negative values indicate convex (dome-shaped, water-shedding) terrain.

## 7. Build the Soil Layer (Sample and Interpolate)

SoilGrids' point-query API allows 5 requests per minute, too slow to query all 48,841 grid cells individually. Instead, this section queries clay content at 25 points spread evenly across the study area, then interpolates between them to estimate a value at every grid cell. This is standard practice when working with a rate-limited point API against a fine target grid, and is documented here as a deliberate methodological choice.

In [9]:
import time
from scipy.interpolate import griddata
import requests
sample_lons = np.linspace(80.310, 80.370, 5)
sample_lats = np.linspace(7.188, 7.248, 5)

sample_points = []
clay_values = []

for lat in sample_lats:
    for lon in sample_lons:
        params = {"lon": lon, "lat": lat, "property": "clay", "depth": "0-5cm", "value": "mean"}

        for attempt in range(2):   # try each point up to twice before giving up on it
            try:
                response = requests.get("https://rest.isric.org/soilgrids/v2.0/properties/query", params=params, timeout=45)
                if response.status_code == 200:
                    raw_value = response.json()["properties"]["layers"][0]["depths"][0]["values"]["mean"]
                    if raw_value is not None:
                        clay_percent = raw_value / 10
                        sample_points.append((lon, lat))
                        clay_values.append(clay_percent)
                        print(f"({lat:.3f}, {lon:.3f}): {clay_percent:.1f}% clay")
                    else:
                        print(f"({lat:.3f}, {lon:.3f}): no data available at this point, skipping")
                else:
                    print(f"({lat:.3f}, {lon:.3f}): request failed, status {response.status_code}")
                break
            except requests.exceptions.RequestException as e:
                print(f"({lat:.3f}, {lon:.3f}): attempt {attempt+1} failed ({type(e).__name__}), {'retrying' if attempt == 0 else 'giving up on this point'}")

        time.sleep(13)

print(f"\n{len(sample_points)} of 25 sample points retrieved successfully")

(7.188, 80.310): 38.5% clay
(7.188, 80.325): 37.5% clay
(7.188, 80.340): attempt 1 failed (ReadTimeout), retrying
(7.188, 80.340): 35.9% clay
(7.188, 80.355): 33.0% clay
(7.188, 80.370): 36.3% clay
(7.203, 80.310): attempt 1 failed (ReadTimeout), retrying
(7.203, 80.310): 33.8% clay
(7.203, 80.325): 37.2% clay
(7.203, 80.340): 33.7% clay
(7.203, 80.355): 34.1% clay
(7.203, 80.370): 38.3% clay
(7.218, 80.310): 33.3% clay
(7.218, 80.325): 33.5% clay
(7.218, 80.340): 35.1% clay
(7.218, 80.355): 36.6% clay
(7.218, 80.370): 38.2% clay
(7.233, 80.310): 35.7% clay
(7.233, 80.325): 37.5% clay
(7.233, 80.340): 36.3% clay
(7.233, 80.355): 35.1% clay
(7.233, 80.370): 38.3% clay
(7.248, 80.310): 35.1% clay
(7.248, 80.325): no data available at this point, skipping
(7.248, 80.340): no data available at this point, skipping
(7.248, 80.355): 35.0% clay
(7.248, 80.370): 35.5% clay

23 of 25 sample points retrieved successfully


This cell takes roughly 5-6 minutes to run, due to the deliberate delay between requests. This is expected and should not be interrupted partway through. A small number of failed points (status other than 200) is tolerable, the interpolation step below only requires enough points to estimate the overall spatial pattern, it does not require all 25 to succeed.

In [10]:
from rasterio.warp import transform as warp_transform

sample_lons_arr = [p[0] for p in sample_points]
sample_lats_arr = [p[1] for p in sample_points]
sample_x, sample_y = warp_transform("EPSG:4326", TARGET_CRS, sample_lons_arr, sample_lats_arr)

rows, cols = np.meshgrid(np.arange(target_height), np.arange(target_width), indexing="ij")
grid_x, grid_y = rasterio.transform.xy(target_transform, rows, cols)
grid_x = np.array(grid_x)
grid_y = np.array(grid_y)

clay_grid = griddata(
    points=list(zip(sample_x, sample_y)),
    values=clay_values,
    xi=(grid_x, grid_y),
    method="linear",
    fill_value=np.mean(clay_values)
)
clay_grid = clay_grid.reshape(target_height, target_width).astype(np.float32)   # restore the 2D grid shape

with rasterio.open(f"{PROCESSED_DIR}/soil_clay_30m.tif", "w", **profile) as dst:
    dst.write(clay_grid, 1)

print(f"Soil clay layer saved")
print(f"Clay content range across study area: {clay_grid.min():.1f}% to {clay_grid.max():.1f}%")

Soil clay layer saved
Clay content range across study area: 33.0% to 38.5%


The resulting range (33.0% to 38.5%) should sit reasonably close to the single-point value of 35.1% confirmed in `01_data_acquisition.ipynb`, since that point falls within the sampled area. Because linear interpolation was used, values should vary smoothly across the grid, without sharp, unrealistic jumps between neighbouring cells. `fill_value` handles the small number of grid corners that may fall outside the convex hull of the sample points, assigning them the area average rather than leaving them empty. 23 of 25 sample points succeeded, with the 2 failures returning a clean "no data" response from ISRIC rather than a connection error, sufficient to estimate a smooth spatial pattern across the study area.